In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
import joblib
import os

In [2]:
columns = [
    'duration','protocol_type','service','flag','src_bytes','dst_bytes',
    'land','wrong_fragment','urgent','hot','num_failed_logins','logged_in',
    'num_compromised','root_shell','su_attempted','num_root',
    'num_file_creations','num_shells','num_access_files','num_outbound_cmds',
    'is_host_login','is_guest_login','count','srv_count','serror_rate',
    'srv_serror_rate','rerror_rate','srv_rerror_rate','same_srv_rate',
    'diff_srv_rate','srv_diff_host_rate','dst_host_count','dst_host_srv_count',
    'dst_host_same_srv_rate','dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate','dst_host_srv_diff_host_rate',
    'dst_host_serror_rate','dst_host_srv_serror_rate','dst_host_rerror_rate',
    'dst_host_srv_rerror_rate','attack_type','difficulty_level'
]

df_train = pd.read_csv('/Users/prashantmeena/Documents/projects/Intrusion Detection System /data/raw/KDDTrain+.txt', header=None, names=columns)
df_test  = pd.read_csv('/Users/prashantmeena/Documents/projects/Intrusion Detection System /data/raw/KDDTest+.txt',  header=None, names=columns)

# drop difficulty_level — not a feature, not a label
df_train.drop('difficulty_level', axis=1, inplace=True)
df_test.drop('difficulty_level',  axis=1, inplace=True)

print(f"Train shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")

Train shape: (125973, 42)
Test shape:  (22544, 42)


In [3]:
attack_map = {
    'normal': 'Normal',
    'back': 'DoS', 'land': 'DoS', 'neptune': 'DoS', 'pod': 'DoS',
    'smurf': 'DoS', 'teardrop': 'DoS', 'apache2': 'DoS',
    'udpstorm': 'DoS', 'processtable': 'DoS', 'worm': 'DoS',
    'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe',
    'satan': 'Probe', 'mscan': 'Probe', 'saint': 'Probe',
    'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L',
    'multihop': 'R2L', 'phf': 'R2L', 'spy': 'R2L',
    'warezclient': 'R2L', 'warezmaster': 'R2L', 'sendmail': 'R2L',
    'named': 'R2L', 'snmpgetattack': 'R2L', 'snmpguess': 'R2L',
    'xlock': 'R2L', 'xsnoop': 'R2L', 'httptunnel': 'R2L',
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R',
    'rootkit': 'U2R', 'ps': 'U2R', 'sqlattack': 'U2R', 'xterm': 'U2R'
}

label_map = {'Normal': 0, 'DoS': 1, 'Probe': 2, 'R2L': 3, 'U2R': 4}

df_train['label'] = df_train['attack_type'].map(attack_map).map(label_map)
df_test['label']  = df_test['attack_type'].map(attack_map).map(label_map)

# drop rows where attack type was unknown (maps to NaN)
df_train.dropna(subset=['label'], inplace=True)
df_test.dropna(subset=['label'],  inplace=True)

df_train['label'] = df_train['label'].astype(int)
df_test['label']  = df_test['label'].astype(int)

print(df_train['label'].value_counts())

label
0    67343
1    45927
2    11656
3      995
4       52
Name: count, dtype: int64


In [4]:
cat_cols = ['protocol_type', 'service', 'flag']
drop_cols = ['attack_type', 'label']

X_train_raw = df_train.drop(drop_cols, axis=1)
X_test_raw  = df_test.drop(drop_cols,  axis=1)

y_train = df_train['label'].values
y_test  = df_test['label'].values

In [5]:
# fit ONLY on train — never fit on test data
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe.fit(X_train_raw[cat_cols])

# transform both train and test using the same fitted encoder
train_cat = ohe.transform(X_train_raw[cat_cols])
test_cat  = ohe.transform(X_test_raw[cat_cols])

# numeric columns (everything except the 3 categorical ones)
num_cols = [c for c in X_train_raw.columns if c not in cat_cols]

train_num = X_train_raw[num_cols].values
test_num  = X_test_raw[num_cols].values

# combine: numeric + one-hot columns
X_train_combined = np.hstack([train_num, train_cat])
X_test_combined  = np.hstack([test_num,  test_cat])

print(f"Feature count after OHE: {X_train_combined.shape[1]}")

Feature count after OHE: 122


In [6]:
# fit ONLY on train
scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(X_train_combined)

X_train_scaled = scaler.transform(X_train_combined)
X_test_scaled  = scaler.transform(X_test_combined)

print(f"Min value in train: {X_train_scaled.min():.3f}")
print(f"Max value in train: {X_train_scaled.max():.3f}")
# both should be 0.0 and 1.0

Min value in train: 0.000
Max value in train: 1.000


In [7]:
# autoencoder trains ONLY on normal traffic (label == 0)
normal_mask = (y_train == 0)
X_train_normal = X_train_scaled[normal_mask]

print(f"Full train set:   {X_train_scaled.shape}")
print(f"Normal-only set:  {X_train_normal.shape}")

Full train set:   (125973, 122)
Normal-only set:  (67343, 122)


In [10]:
os.makedirs('../data/processed', exist_ok=True)

np.save('../data/processed/X_train.npy', X_train_scaled)
np.save('../data/processed/X_test.npy',  X_test_scaled)
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/y_test.npy',  y_test)
np.save('../data/processed/X_train_normal.npy', X_train_normal)

# save the fitted encoder and scaler together
joblib.dump({'ohe': ohe, 'scaler': scaler, 'num_cols': num_cols},
            '../data/processed/encoder_artifacts.pkl')

print("All files saved to data/processed/")

All files saved to data/processed/


In [11]:
# sanity check — reload and confirm shapes
X_tr = np.load('../data/processed/X_train.npy')
X_te = np.load('../data/processed/X_test.npy')
y_tr = np.load('../data/processed/y_train.npy')
y_te = np.load('../data/processed/y_test.npy')
X_normal = np.load('../data/processed/X_train_normal.npy')

print(f"X_train:       {X_tr.shape}")
print(f"X_test:        {X_te.shape}")
print(f"y_train:       {y_tr.shape}")
print(f"y_test:        {y_te.shape}")
print(f"X_train_normal:{X_normal.shape}")

X_train:       (125973, 122)
X_test:        (22251, 122)
y_train:       (125973,)
y_test:        (22251,)
X_train_normal:(67343, 122)
